# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
# REMOVE API keys before pushing code to Github.
key = None
source = None

# 1. Colab Secrets
try:
    from google.colab import userdata
    for _name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        try:
            key = userdata.get(_name)
        except Exception:
            key = None
        if key:
            source = "Colab Secrets"
            break
except Exception:
    pass

# 2. Environment variables (already exported on this machine)
if not key:
    key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if key:
        source = "environment"

# 3. Manual entry if Secrets / env are not set up. This fallback helps.
if not key:
    import getpass
    key = getpass.getpass("Paste your Gemini API key and press Enter: ")
    source = "manual entry"

if not key:
    raise ValueError(
        "No Gemini API key found. Set GOOGLE_API_KEY or GEMINI_API_KEY, "
        "or paste it when prompted."
    )

# Both libraries look for one of these names.
os.environ["GOOGLE_API_KEY"] = key
os.environ["GEMINI_API_KEY"] = key
print(f"API key loaded from {source}.")


API key loaded from environment.


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai google-genai --quiet
print("Done.")


Done.


In [3]:
# @title User input needed.

DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section."
        "Documents can be related to any topic - sports, manga, food, music etc."
        "Have a minimum of 4 sentences with each document."
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),
    "fill_in1": (
        "In this fictional example, the Riverside cricket club practices every Saturday at 7 AM. "
        "The team captain is Maya, and the coach is Arun. "
        "Players spend the first thirty minutes warming up and practicing catches. "
        "Their next match is against the Hilltop club at Greenfield Stadium."
    ),
    "fill_in2": (
        "In the fictional manga Star Lantern, a student named Riku finds a glowing map. "
        "The map leads him to a hidden library beneath his school. "
        "His friend Hana helps him solve puzzles using clues from old books. "
        "Their goal is to recover three lost lanterns before the winter festival."
    ),
    "fill_in3": (
        "This sample recipe makes a simple tomato pasta for two people. "
        "The ingredients are 200 grams of pasta, three tomatoes, two garlic cloves, and one tablespoon of olive oil. "
        "Boil the pasta according to its packet instructions and cook the chopped tomatoes with garlic in olive oil. "
        "Mix the pasta with the sauce and serve it with fresh basil."
    ),
    "fill_in4": (
        "In this fictional example, the band Midnight Mango has four members. "
        "Leena sings, Dev plays guitar, Noor plays drums, and Kabir plays bass. "
        "The band rehearses on Wednesday evenings in a community studio. "
        "Their debut album is called Paper Moons and contains eight songs."
    )
}

In [4]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []
    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for document search.
print(search_documents("minimum of 4"))


[Source: sample_protocol]
Students must fill in documents in this section.Documents can be related to any topic - sports, manga, food, music etc.Have a minimum of 4 sentences with each document.Duplicate this for a minimum of 4 documents to test your agent.

[Source: fill_in3]
This sample recipe makes a simple tomato pasta for two people. The ingredients are 200 grams of pasta, three tomatoes, two garlic cloves, and one tablespoon of olive oil. Boil the pasta according to its packet instructions and cook the chopped tomatoes with garlic in olive oil. Mix the pasta with the sauce and serve it with fresh basil.


In [5]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)


Model ready: models/gemini-3.5-flash-lite


In [6]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Question: What topics are students supposed to share documents on?
--------------------------------------------------


Step 1: model calls search_documents({'query': 'share documents topics students'})
  Result preview: [Source: sample_protocol]
Students must fill in documents in this section.Documents can be related t...



Final answer:
Students are supposed to share documents related to any topic of their choice, such as sports, manga, food, music, etc.


In [7]:
# @title Ask follow up questions

# Questions grounded in DOCUMENTS, plus two checks the agent should handle carefully.
questions = [
    "Who is the captain of the Riverside cricket club, and when does the team practice?",
    "In the manga Star Lantern, what does Riku find, and what is his goal?",
    "What ingredients are needed for the tomato pasta recipe, and how many people does it serve?",
    "How many members does Midnight Mango have, and what is their debut album called?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?",
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Who is the captain of the Riverside cricket club, and when does the team practice?
--------------------------------------------------


Step 1: model calls search_documents({'query': 'Riverside cricket club captain practice'})
  Result preview: [Source: fill_in1]
In this fictional example, the Riverside cricket club practices every Saturday at...



Final answer:
The captain of the Riverside cricket club is **Maya**, and the team practices **every Saturday at 7 AM**.



Question: In the manga Star Lantern, what does Riku find, and what is his goal?
--------------------------------------------------


Step 1: model calls search_documents({'query': 'Star Lantern Riku find goal'})
  Result preview: [Source: fill_in2]
In the fictional manga Star Lantern, a student named Riku finds a glowing map. Th...



Final answer:
In the manga *Star Lantern*, Riku finds a glowing map that leads him to a hidden library beneath his school. His goal is to recover three lost lanterns before the winter festival.


... (11 outputs left out)


Question: What is the most popular food item in Canada?
--------------------------------------------------


Step 1: model calls search_documents({'query': 'most popular food item in Canada'})
  Result preview: [Source: sample_protocol]
Students must fill in documents in this section.Documents can be related t...


Step 2: model calls search_documents({'query': 'poutine canada food'})
  Result preview: [Source: sample_protocol]
Students must fill in documents in this section.Documents can be related t...



Final answer:
Based on the available documents, there is no information regarding the most popular food item in Canada.


In [8]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)